# Group Theory
## From Symmetries to Structure — An Algebraic Exploration

---

**Topics:** Group axioms · Subgroups · Cayley tables · Homomorphisms · Lagrange's theorem · Permutation groups · Connection to Lie groups

**Prerequisites:** Basic set theory, linear algebra (matrices), modular arithmetic

**Key References:**
- Dummit & Foote, *Abstract Algebra* (3rd ed.), Chapters 1–3
- Artin, *Algebra* (2nd ed.), Chapters 2, 6
- Hall, *The Theory of Groups*
- See also: `../lie-groups/lie_groups.ipynb` for the continuous-group extension

## 1. Problem Statement — What Are Groups and Why Do They Matter?

A **group** is the mathematical distillation of *symmetry*. Whenever a system is unchanged under a collection of transformations that can be composed and reversed, those transformations form a group.

### Why groups appear everywhere

| Domain | Group | What it encodes |
|--------|-------|-----------------|
| Geometry | $SO(3)$ | 3-D rotations of rigid bodies |
| Physics | $SU(2)$, $SU(3)$ | Particle symmetries (Standard Model) |
| Cryptography | $(\mathbb{Z}_n, +)$, elliptic curves | Discrete-log hardness |
| Robotics | $SE(3)$ | Poses of a robot in space |
| Computer vision | Permutation groups | Graph isomorphism, image symmetry |
| Coding theory | $\mathbb{F}_2^n$ | Error-correcting codes |

### Roadmap of this notebook

$$\underbrace{\text{Axioms}}_\text{foundations} \;\longrightarrow\; \underbrace{\text{Examples}}_\text{intuition} \;\longrightarrow\; \underbrace{\text{Subgroups & Cayley tables}}_\text{structure} \;\longrightarrow\; \underbrace{\text{Homomorphisms & Lagrange}}_\text{theory} \;\longrightarrow\; \underbrace{\text{Permutations}}_\text{S_n} \;\longrightarrow\; \underbrace{\text{Lie groups}}_\text{continuous}$$

In [ ]:
%matplotlib inline

import itertools
import warnings
from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as pe

warnings.filterwarnings('ignore')

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize' : (10, 5),
    'font.size'      : 12,
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
    'lines.linewidth': 2,
})

COLORS = {
    'blue'   : '#2196F3',
    'orange' : '#FF9800',
    'green'  : '#4CAF50',
    'red'    : '#F44336',
    'purple' : '#9C27B0',
    'teal'   : '#009688',
    'amber'  : '#FFC107',
    'indigo' : '#3F51B5',
}

print("Imports OK — NumPy", np.__version__)

## 2. Group Axioms

A **group** $(G, \star)$ is a set $G$ together with a binary operation $\star : G \times G \to G$ satisfying four axioms:

$$\boxed{
\begin{aligned}
&\textbf{(G1) Closure:}       && \forall\, a,b\in G,\quad a\star b\in G \\
&\textbf{(G2) Associativity:} && \forall\, a,b,c\in G,\quad (a\star b)\star c = a\star(b\star c) \\
&\textbf{(G3) Identity:}      && \exists\, e\in G \text{ s.t. } \forall\, a\in G,\; e\star a = a\star e = a \\
&\textbf{(G4) Inverse:}       && \forall\, a\in G,\; \exists\, a^{-1}\in G \text{ s.t. } a\star a^{-1} = a^{-1}\star a = e
\end{aligned}
}$$

If additionally $a \star b = b \star a$ for all $a, b \in G$, the group is **abelian** (commutative).

The **order** $|G|$ is the cardinality of the set $G$. A group of finite order is called a *finite group*.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Core Group class — from scratch
# ═══════════════════════════════════════════════════════════════════════════════

class Group:
    """
    Represents a finite group via an explicit multiplication table.

    Parameters
    ----------
    elements : list
        Ordered list of group elements (any hashable objects).
    op : callable
        Binary operation op(a, b) -> element in elements.
    name : str
        Human-readable name.
    """

    def __init__(self, elements, op, name='G'):
        self.elements = list(elements)
        self.op       = op
        self.name     = name
        self.n        = len(elements)
        self._idx     = {e: i for i, e in enumerate(elements)}
        self._table   = self._build_table()

    # ── internal helpers ──────────────────────────────────────────────────────
    def _build_table(self):
        """Build the Cayley table as an (n x n) index array."""
        n   = self.n
        tbl = np.zeros((n, n), dtype=int)
        for i, a in enumerate(self.elements):
            for j, b in enumerate(self.elements):
                result = self.op(a, b)
                if result not in self._idx:
                    raise ValueError(f"Closure violated: {a} * {b} = {result} not in G")
                tbl[i, j] = self._idx[result]
        return tbl

    # ── public API ────────────────────────────────────────────────────────────
    def mul(self, a, b):
        """Multiply two elements."""
        return self.elements[self._table[self._idx[a], self._idx[b]]]

    def identity(self):
        """Return the identity element."""
        for e in self.elements:
            if all(self.mul(e, a) == a and self.mul(a, e) == a
                   for a in self.elements):
                return e
        return None

    def inverse(self, a):
        """Return the inverse of element a."""
        e = self.identity()
        for b in self.elements:
            if self.mul(a, b) == e:
                return b
        return None

    def order_of(self, a):
        """Return the order of element a (smallest k>0 with a^k = e)."""
        e   = self.identity()
        cur = a
        for k in range(1, self.n + 1):
            if cur == e:
                return k
            cur = self.mul(cur, a)
        return None  # should never happen in a valid finite group

    def is_abelian(self):
        """Check if the group is commutative."""
        return np.array_equal(self._table, self._table.T)

    def verify_axioms(self, verbose=True):
        """
        Verify all four group axioms.

        Returns
        -------
        dict with keys 'closure', 'associativity', 'identity', 'inverse'
        """
        results = {}

        # G1: closure — guaranteed by table construction; just confirm no KeyError
        try:
            _ = self._table
            results['closure'] = True
        except Exception:
            results['closure'] = False

        # G2: associativity
        assoc = True
        for a in self.elements:
            for b in self.elements:
                for c in self.elements:
                    if self.mul(self.mul(a, b), c) != self.mul(a, self.mul(b, c)):
                        assoc = False
                        break
        results['associativity'] = assoc

        # G3: identity
        e = self.identity()
        results['identity'] = (e is not None)

        # G4: inverse
        results['inverse'] = all(self.inverse(a) is not None for a in self.elements)

        if verbose:
            print(f"\nAxiom verification for {self.name} (order {self.n}):")
            for ax, ok in results.items():
                tag = '[PASS]' if ok else '[FAIL]'
                print(f"  {tag}  {ax.capitalize()}")
            if all(results.values()):
                print(f"  => Valid group.  Abelian: {self.is_abelian()}")

        return results

    def subgroups(self):
        """Return all subgroups (as lists of elements) via brute-force subset check."""
        e    = self.identity()
        subs = []
        # iterate over all subsets that contain e
        for r in range(1, self.n + 1):
            for subset in itertools.combinations(self.elements, r):
                subset = list(subset)
                if e not in subset:
                    continue
                # closure under op
                closed = all(self.mul(a, b) in subset
                             for a in subset for b in subset)
                if not closed:
                    continue
                # inverse
                has_inv = all(self.inverse(a) in subset for a in subset)
                if has_inv:
                    subs.append(subset)
        return subs

    def cosets(self, H_elements):
        """
        Return the left cosets of a subgroup H.
        H_elements: list of elements forming a subgroup.
        """
        H      = list(H_elements)
        cosets = []
        seen   = set()
        for a in self.elements:
            aH = tuple(sorted(str(self.mul(a, h)) for h in H))
            if aH not in seen:
                seen.add(aH)
                cosets.append([self.mul(a, h) for h in H])
        return cosets

print("Group class defined.")

## 3. Examples of Groups

### 3.1 Integers under addition $(\mathbb{Z}, +)$

- **Closure:** $m + n \in \mathbb{Z}$
- **Identity:** $0$
- **Inverse:** $-n$
- **Abelian:** yes, infinite order

### 3.2 Modular arithmetic $(\mathbb{Z}_n, +_n)$

$$a +_n b := (a + b) \bmod n$$

This is a *finite* abelian group of order $n$. Every element $a \in \mathbb{Z}_n$ has order $n / \gcd(a, n)$.

### 3.3 Symmetric / permutation groups $S_n$

$S_n$ is the group of all bijections $\sigma : \{1,\ldots,n\} \to \{1,\ldots,n\}$ under function composition. $|S_n| = n!$. For $n \geq 3$, $S_n$ is **non-abelian**.

### 3.4 Matrix groups

| Symbol | Definition | Operation | Order |
|--------|-----------|-----------|-------|
| $GL(n,\mathbb{R})$ | $n\times n$ invertible real matrices | matrix multiplication | $\infty$ |
| $SL(n,\mathbb{R})$ | $\det(A)=1$ | matrix multiplication | $\infty$ |
| $O(n)$ | $A^\top A = I$ | matrix multiplication | $\infty$ |
| $SO(n)$ | $O(n)$ with $\det(A)=+1$ | matrix multiplication | $\infty$ |

All are subgroups of $GL(n,\mathbb{R})$: $SO(n) \subset O(n) \subset GL(n,\mathbb{R})$.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Construct and verify concrete small groups
# ═══════════════════════════════════════════════════════════════════════════════

# ── (Z_6, +6) ─────────────────────────────────────────────────────────────────
Z6 = Group(
    elements=list(range(6)),
    op=lambda a, b: (a + b) % 6,
    name='(Z_6, +_6)'
)
res_Z6 = Z6.verify_axioms()

print("\nElement orders in Z_6:")
for a in Z6.elements:
    print(f"  ord({a}) = {Z6.order_of(a)}")

# ── Matrix groups: check O(2) and SO(2) membership ────────────────────────────
print("\nMatrix group membership checks:")
thetas = [0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2]
for t in thetas:
    R = np.array([[np.cos(t), -np.sin(t)],
                  [np.sin(t),  np.cos(t)]])
    in_O2  = np.allclose(R.T @ R, np.eye(2))
    in_SO2 = in_O2 and np.isclose(np.linalg.det(R), 1.0)
    print(f"  theta={np.degrees(t):6.1f}°  O(2):{in_O2}  SO(2):{in_SO2}")

# Reflection is O(2) but not SO(2)
Reflect = np.array([[1, 0], [0, -1]])
in_O2_r  = np.allclose(Reflect.T @ Reflect, np.eye(2))
in_SO2_r = in_O2_r and np.isclose(np.linalg.det(Reflect), 1.0)
print(f"  Reflection matrix  O(2):{in_O2_r}  SO(2):{in_SO2_r}  det={np.linalg.det(Reflect):.0f}")

## 4. Subgroups

A subset $H \subseteq G$ is a **subgroup** (written $H \leq G$) if and only if $(H, \star)$ is itself a group.

### One-step subgroup test

$$\boxed{H \leq G \iff H \neq \emptyset \text{ and } \forall\, a, b \in H : a \star b^{-1} \in H}$$

### Examples in $S_3$

$S_3$ has order $6$ and its subgroup lattice is:
- Trivial subgroup $\{e\}$ (order 1)
- Three cyclic subgroups of order 2: $\{e,(12)\}$, $\{e,(13)\}$, $\{e,(23)\}$
- One cyclic subgroup of order 3: $A_3 = \{e,(123),(132)\}$ (alternating group)
- The full group $S_3$ (order 6)

Note: $A_3$ is the unique *normal* subgroup of $S_3$ of index 2.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Build S_3 and enumerate all subgroups
# ═══════════════════════════════════════════════════════════════════════════════

def perm_compose(sigma, tau):
    """Compose permutations given as tuples: sigma ∘ tau (apply tau first)."""
    return tuple(sigma[tau[i]] for i in range(len(sigma)))

# Elements of S_3 as tuples (0-indexed)
S3_elements = list(itertools.permutations(range(3)))

S3 = Group(
    elements=S3_elements,
    op=perm_compose,
    name='S_3'
)
res_S3 = S3.verify_axioms()

# ── label permutations in standard cycle-notation-like names ─────────────────
S3_labels = {
    (0,1,2): 'e',
    (1,2,0): '(123)',
    (2,0,1): '(132)',
    (0,2,1): '(23)',
    (2,1,0): '(13)',
    (1,0,2): '(12)',
}

print("\nSubgroups of S_3:")
subs = S3.subgroups()
for s in subs:
    names = [S3_labels.get(x, str(x)) for x in s]
    print(f"  order {len(s):2d}  {sorted(names)}")

# ── visualize subgroup lattice ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.3, 3.3)
ax.axis('off')
ax.set_title('Subgroup Lattice of $S_3$', fontsize=14, fontweight='bold')

# node positions  (x, y) — three levels
nodes = {
    'S_3'    : (2.0, 3.0),
    'A_3'    : (2.0, 2.0),
    '⟨(12)⟩' : (0.5, 2.0),
    '⟨(13)⟩' : (2.0, 1.0),
    '⟨(23)⟩' : (3.5, 2.0),
    '{e}'    : (2.0, 0.0),
}

# edges in the lattice (child -> parent)
edges = [
    ('{e}',    'A_3'),
    ('{e}',    '⟨(12)⟩'),
    ('{e}',    '⟨(13)⟩'),
    ('{e}',    '⟨(23)⟩'),
    ('A_3',    'S_3'),
    ('⟨(12)⟩', 'S_3'),
    ('⟨(13)⟩', 'S_3'),
    ('⟨(23)⟩', 'S_3'),
]

node_colors = {
    'S_3'    : COLORS['blue'],
    'A_3'    : COLORS['green'],
    '⟨(12)⟩' : COLORS['orange'],
    '⟨(13)⟩' : COLORS['orange'],
    '⟨(23)⟩' : COLORS['orange'],
    '{e}'    : COLORS['red'],
}

for (src, dst) in edges:
    x0, y0 = nodes[src]
    x1, y1 = nodes[dst]
    ax.plot([x0, x1], [y0, y1], 'k-', lw=1.5, zorder=1, alpha=0.5)

for label, (x, y) in nodes.items():
    ax.scatter(x, y, s=900, color=node_colors[label], zorder=3, edgecolors='k', linewidths=1.2)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5,
            fontweight='bold', color='white', zorder=4)

# legend
legend_items = [
    mpatches.Patch(color=COLORS['blue'],   label='$S_3$ (order 6)'),
    mpatches.Patch(color=COLORS['green'],  label='$A_3$ (order 3, normal)'),
    mpatches.Patch(color=COLORS['orange'], label='Order-2 subgroups'),
    mpatches.Patch(color=COLORS['red'],    label='Trivial $\\{e\\}$'),
]
ax.legend(handles=legend_items, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Cayley Tables

The **Cayley table** (or multiplication table) of a finite group $G = \{g_1, \ldots, g_n\}$ is the $n \times n$ matrix whose $(i,j)$ entry is $g_i \star g_j$.

### Key structural properties readable from the table

- **Latin square property:** each element appears exactly once in every row and column (direct consequence of the cancellation law).
- **Symmetry about diagonal $\Leftrightarrow$ abelian group.**
- **Identity row/column** is a fixed row/column of the table.

We compare three groups of order 4:

| Group | Abelian? | Cyclic? | Description |
|-------|----------|---------|-------------|
| $\mathbb{Z}_4$ | yes | yes | cyclic group of order 4 |
| $V_4$ (Klein four-group) | yes | no | $\mathbb{Z}_2 \times \mathbb{Z}_2$ |

And the non-abelian group $S_3$ (order 6).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cayley table computation and heatmap visualization
# ═══════════════════════════════════════════════════════════════════════════════

def plot_cayley_table(group, labels, ax, title=''):
    """
    Draw the Cayley table of a Group as a coloured heatmap.

    Parameters
    ----------
    group  : Group instance
    labels : list[str]  — display name for each element
    ax     : matplotlib Axes
    title  : str
    """
    n   = group.n
    tbl = group._table  # n×n index array

    cmap = plt.cm.get_cmap('tab20', n)
    ax.imshow(tbl, cmap=cmap, vmin=0, vmax=n - 1, origin='upper')

    # cell annotations
    for i in range(n):
        for j in range(n):
            ax.text(j, i, labels[tbl[i, j]],
                    ha='center', va='center', fontsize=11,
                    fontweight='bold', color='white',
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='black')])

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.xaxis.tick_top()
    ax.set_title(title, fontsize=12, fontweight='bold', pad=14)
    # draw grid
    for k in range(n + 1):
        ax.axhline(k - 0.5, color='white', lw=0.8)
        ax.axvline(k - 0.5, color='white', lw=0.8)


# ── Define the three groups ───────────────────────────────────────────────────
Z4 = Group(
    elements=list(range(4)),
    op=lambda a, b: (a + b) % 4,
    name='Z_4'
)

# Klein four-group V4: elements {e, a, b, c} with a²=b²=c²=e, ab=c, etc.
V4_elems = ['e', 'a', 'b', 'c']
V4_mul = {
    ('e','e'):'e', ('e','a'):'a', ('e','b'):'b', ('e','c'):'c',
    ('a','e'):'a', ('a','a'):'e', ('a','b'):'c', ('a','c'):'b',
    ('b','e'):'b', ('b','a'):'c', ('b','b'):'e', ('b','c'):'a',
    ('c','e'):'c', ('c','a'):'b', ('c','b'):'a', ('c','c'):'e',
}
V4 = Group(elements=V4_elems, op=lambda a, b: V4_mul[(a, b)], name='V_4 (Klein)')

# ── Verify and plot ───────────────────────────────────────────────────────────
print("Verifying Z_4:")
Z4.verify_axioms()
print("\nVerifying V_4 (Klein four-group):")
V4.verify_axioms()
print("\nVerifying S_3 (already done, re-checking abelian):")
print(f"  S_3 abelian: {S3.is_abelian()}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

plot_cayley_table(Z4, [str(x) for x in Z4.elements], axes[0], title='$\\mathbb{Z}_4$ (cyclic, abelian)')
plot_cayley_table(V4, V4.elements, axes[1], title='$V_4$ (Klein four-group, abelian)')
plot_cayley_table(S3, [S3_labels[x] for x in S3.elements], axes[2], title='$S_3$ (non-abelian)')

plt.suptitle('Cayley Tables — Latin Square Property', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nLatin square check (each element appears exactly once per row/col):")
for G in [Z4, V4, S3]:
    tbl = G._table
    ok  = all(len(set(tbl[i])) == G.n for i in range(G.n)) and \
          all(len(set(tbl[:, j])) == G.n for j in range(G.n))
    print(f"  {G.name:20s}  [{'PASS' if ok else 'FAIL'}]")

## 6. Homomorphisms and Isomorphisms

### Definition

A map $\phi : (G, \star) \to (H, \cdot)$ is a **group homomorphism** if

$$\boxed{\phi(a \star b) = \phi(a) \cdot \phi(b) \quad \forall\, a, b \in G}$$

Consequences: $\phi(e_G) = e_H$ and $\phi(a^{-1}) = \phi(a)^{-1}$.

### Kernel and Image

$$\ker\phi = \{g \in G : \phi(g) = e_H\} \leq G \quad\text{(normal subgroup)}$$
$$\mathrm{im}\,\phi = \{\phi(g) : g \in G\} \leq H$$

### First Isomorphism Theorem

$$\boxed{G / \ker\phi \;\cong\; \mathrm{im}\,\phi}$$

This is one of the most powerful structural results in group theory: it says the quotient of $G$ by the kernel is *isomorphic* to the image.

An **isomorphism** is a bijective homomorphism. Two groups are isomorphic ($G \cong H$) if and only if they are structurally identical — same Cayley table up to relabelling of elements.

### Example

The sign map $\mathrm{sgn} : S_3 \to (\{\pm 1\}, \times)$ defined by $\sigma \mapsto (-1)^{\text{\# inversions}}$ is a surjective homomorphism with $\ker(\mathrm{sgn}) = A_3$, giving $S_3 / A_3 \cong \mathbb{Z}_2$.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Verify a homomorphism: sign map  sgn : S_3 -> {+1, -1}
# ═══════════════════════════════════════════════════════════════════════════════

def sign_of_permutation(sigma):
    """
    Compute the sign (+1 or -1) of permutation sigma (tuple, 0-indexed).
    Uses the inversion count: inversions = pairs (i,j) with i<j but sigma[i]>sigma[j].
    """
    n   = len(sigma)
    inv = sum(1 for i in range(n) for j in range(i+1, n) if sigma[i] > sigma[j])
    return (-1) ** inv

print("Sign map on S_3 elements:")
for perm in S3.elements:
    label = S3_labels[perm]
    sgn   = sign_of_permutation(perm)
    print(f"  sgn({label:7s}) = {sgn:+d}")

# Verify homomorphism property: sgn(sigma ∘ tau) = sgn(sigma) * sgn(tau)
print("\nHomomorphism check sgn(a*b) == sgn(a)*sgn(b):")
all_ok = True
for a in S3.elements:
    for b in S3.elements:
        ab  = S3.mul(a, b)
        lhs = sign_of_permutation(ab)
        rhs = sign_of_permutation(a) * sign_of_permutation(b)
        if lhs != rhs:
            all_ok = False
            print(f"  FAIL: {S3_labels[a]} * {S3_labels[b]}")
print(f"  [{'PASS' if all_ok else 'FAIL'}]  sgn is a valid homomorphism")

# Kernel
kernel = [p for p in S3.elements if sign_of_permutation(p) == 1]
print(f"\n  ker(sgn) = A_3 = {sorted(S3_labels[p] for p in kernel)}")
print(f"  |S_3 / ker(sgn)| = {S3.n // len(kernel)}  (=|Z_2|)  [First Iso. Theorem]")

# ── Check Z_4 and Z_2 x Z_2 are NOT isomorphic (different element orders) ─────
print("\nIsomorphism check — Z_4 vs V_4:")
print(f"  Element orders in Z_4: {sorted(Z4.order_of(a) for a in Z4.elements)}")
print(f"  Element orders in V_4: {sorted(V4.order_of(a) for a in V4.elements)}")
print("  => Different order multisets  =>  Z_4 is NOT isomorphic to V_4")

## 7. Lagrange's Theorem

### Statement

$$\boxed{\text{If } H \leq G \text{ and } |G| < \infty, \text{ then } |H| \text{ divides } |G|.}$$

### Proof sketch

The left cosets $\{aH : a \in G\}$ **partition** $G$ and every coset has exactly $|H|$ elements (left-multiplication by $a$ is a bijection $H \to aH$). If there are $[G:H]$ distinct cosets, then:

$$|G| = [G:H] \cdot |H|$$

The integer $[G:H]$ is called the **index** of $H$ in $G$.

### Corollaries

1. **Order of an element divides $|G|$:** For any $g \in G$, $\mathrm{ord}(g)$ divides $|G|$ (because $\langle g \rangle \leq G$).
2. **Groups of prime order are cyclic:** If $|G| = p$ is prime, $G \cong \mathbb{Z}_p$.
3. **Fermat's little theorem** is a corollary: $a^p \equiv a \pmod{p}$ for prime $p$.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Verify Lagrange's theorem for all subgroups of S_3
# ═══════════════════════════════════════════════════════════════════════════════

print(f"Lagrange's Theorem verification for {S3.name} (|G| = {S3.n}):")
print(f"{'Subgroup':<28}  |H|  |G|/|H|  Divides?  Index")
print("-" * 60)

for H in S3.subgroups():
    h_labels = sorted(S3_labels[x] for x in H)
    h_order  = len(H)
    divides  = (S3.n % h_order == 0)
    index    = S3.n // h_order if divides else 'N/A'
    cosets   = S3.cosets(H)
    print(f"  {str(h_labels):<26}  {h_order:3d}  {S3.n}/{h_order:1d}={index}   "
          f"  [{'PASS' if divides else 'FAIL'}]    {index}")

# ── Corollary 1: element orders divide |G| ────────────────────────────────────
print(f"\nCorollary: element orders divide |S_3| = {S3.n}")
for perm in S3.elements:
    ord_a  = S3.order_of(perm)
    label  = S3_labels[perm]
    divides= (S3.n % ord_a == 0)
    print(f"  ord({label:7s}) = {ord_a}   {S3.n} % {ord_a} = {S3.n % ord_a}  [{'PASS' if divides else 'FAIL'}]")

# ── Corollary 2: groups of prime order ────────────────────────────────────────
Z5 = Group(elements=list(range(5)), op=lambda a, b: (a + b) % 5, name='Z_5')
sub_Z5 = Z5.subgroups()
print(f"\nSubgroups of Z_5 (prime order 5): {[len(s) for s in sub_Z5]} — trivial and full only")

## 8. Permutation Groups — Cycle Notation and Parity

### Cycle notation

A permutation $\sigma \in S_n$ can be written as a product of disjoint cycles. For example in $S_4$:

$$\sigma = \begin{pmatrix}1&2&3&4\\2&4&1&3\end{pmatrix} = (1\;2\;4\;3)$$

### Composition

Convention: $(\sigma \circ \tau)(x) = \sigma(\tau(x))$ — apply $\tau$ first.

### Even and odd permutations

Every permutation is a product of transpositions. The **parity** (even/odd) is well-defined:

$$\boxed{\text{sgn}(\sigma) = (-1)^{\text{\# inversions}} \in \{+1, -1\}}$$

The **alternating group** $A_n = \ker(\text{sgn}) = \{\sigma \in S_n : \text{sgn}(\sigma) = +1\}$ has order $n!/2$.

### Cayley's Theorem

Every finite group $G$ is isomorphic to a subgroup of $S_{|G|}$. Permutation groups are therefore universal.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Permutation utilities — cycle notation, composition, parity
# ═══════════════════════════════════════════════════════════════════════════════

def to_cycle_notation(sigma):
    """
    Convert a permutation tuple (0-indexed) to disjoint cycle notation string.
    Fixed points are omitted.
    """
    n       = len(sigma)
    visited = [False] * n
    cycles  = []
    for start in range(n):
        if visited[start] or sigma[start] == start:
            visited[start] = True
            continue
        cycle   = []
        current = start
        while not visited[current]:
            visited[current] = True
            cycle.append(current + 1)  # 1-indexed display
            current = sigma[current]
        if len(cycle) > 1:
            cycles.append('(' + ' '.join(map(str, cycle)) + ')')
    return ''.join(cycles) if cycles else 'e'


def cycle_type(sigma):
    """Return sorted tuple of cycle lengths (including fixed points as 1-cycles)."""
    n       = len(sigma)
    visited = [False] * n
    lengths = []
    for start in range(n):
        if visited[start]:
            continue
        length  = 0
        current = start
        while not visited[current]:
            visited[current] = True
            current = sigma[current]
            length += 1
        lengths.append(length)
    return tuple(sorted(lengths, reverse=True))


# ── Demonstrate on S_4 ────────────────────────────────────────────────────────
S4_elements = list(itertools.permutations(range(4)))
S4 = Group(elements=S4_elements, op=perm_compose, name='S_4')

print(f"All permutations of S_4 (order {S4.n} = 4!):")
print(f"  {'Tuple':15s}  {'Cycle notation':15s}  {'Cycle type':12s}  {'Sign':5s}  {'Parity'}")
print("  " + "-" * 65)
for perm in S4_elements[:12]:   # show first 12 for readability
    cn    = to_cycle_notation(perm)
    ct    = cycle_type(perm)
    sgn   = sign_of_permutation(perm)
    parity= 'even' if sgn == 1 else 'odd'
    print(f"  {str(perm):15s}  {cn:15s}  {str(ct):12s}  {sgn:+2d}    {parity}")

# Alternating group A_4
A4_elements = [p for p in S4_elements if sign_of_permutation(p) == 1]
print(f"\n|A_4| = {len(A4_elements)} = {S4.n}!/2")

# ── Verify: composition of two odd permutations is even ───────────────────────
t1 = (1, 0, 2, 3)  # transposition (1 2)
t2 = (0, 2, 1, 3)  # transposition (2 3)
comp = perm_compose(t1, t2)
print(f"\nCompose two odd perms: sgn({to_cycle_notation(t1)}) * sgn({to_cycle_notation(t2)}) = "
      f"{sign_of_permutation(t1)} * {sign_of_permutation(t2)} = {sign_of_permutation(t1)*sign_of_permutation(t2)}")
print(f"  Result {to_cycle_notation(comp)} has sign {sign_of_permutation(comp)}  [PASS]")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Visualize cycle types and parity structure of S_4
# ═══════════════════════════════════════════════════════════════════════════════

from collections import Counter

cycle_type_counts = Counter(cycle_type(p) for p in S4_elements)
parity_counts     = Counter('even' if sign_of_permutation(p) == 1 else 'odd'
                            for p in S4_elements)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: cycle type distribution
ax = axes[0]
ct_labels = [str(k) for k in sorted(cycle_type_counts)]
ct_values = [cycle_type_counts[k] for k in sorted(cycle_type_counts)]
bar_colors= [COLORS['blue'], COLORS['orange'], COLORS['green'],
             COLORS['red'], COLORS['purple']]
bars = ax.bar(ct_labels, ct_values, color=bar_colors[:len(ct_labels)],
              edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, ct_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Cycle type', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Cycle Types in $S_4$', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.4)

# Right: parity pie
ax2 = axes[1]
wedge_colors = [COLORS['green'], COLORS['red']]
parity_labels_list = ['even', 'odd']
sizes = [parity_counts[k] for k in parity_labels_list]
wedges, texts, autotexts = ax2.pie(
    sizes, labels=[f'{k}\n({v})' for k, v in zip(parity_labels_list, sizes)],
    colors=wedge_colors, autopct='%1.0f%%', startangle=90,
    textprops={'fontsize': 11}, pctdistance=0.6
)
for at in autotexts:
    at.set_fontweight('bold')
ax2.set_title('Parity distribution in $S_4$\n(A_4 = even permutations)', fontsize=12, fontweight='bold')

plt.suptitle('Permutation Structure of $S_4$', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nCycle type → conjugacy class summary:")
parity_map = {ct: ('even' if sum(l-1 for l in ct) % 2 == 0 else 'odd')
              for ct in sorted(cycle_type_counts)}
print(f"  {'Cycle type':12s}  {'Count':6s}  {'Parity'}")
for ct in sorted(cycle_type_counts):
    print(f"  {str(ct):12s}  {cycle_type_counts[ct]:6d}  {parity_map[ct]}")

## 9. Connection to Lie Groups

The groups considered above are *discrete* (finite or countably infinite). A **Lie group** is a group that is simultaneously a smooth manifold, with the group operations smooth maps. This is the bridge from abstract algebra to differential geometry.

### SO(2) — the simplest compact Lie group

$$SO(2) = \left\{\, R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{pmatrix} : \theta \in \mathbb{R}\,\right\}$$

This is a *continuous* group: $\theta$ is a real parameter. The group operation is matrix multiplication, and $SO(2) \cong (\mathbb{R}/2\pi\mathbb{Z},\, +)$.

### The Lie algebra $\mathfrak{so}(2)$ and the exponential map

The **Lie algebra** is the tangent space at the identity, endowed with the Lie bracket $[A,B] = AB - BA$:

$$\mathfrak{so}(2) = \left\{\, \begin{pmatrix}0 & -\omega \\ \omega & 0\end{pmatrix} : \omega \in \mathbb{R}\,\right\}$$

The **exponential map** connects algebra to group:

$$\boxed{\exp : \mathfrak{so}(2) \to SO(2), \qquad \exp\!\left(\omega\begin{pmatrix}0&-1\\1&0\end{pmatrix}\right) = R(\omega)}$$

This generalises to $SO(3)$, $SE(3)$, and all matrix Lie groups used in robotics.

> **See also:** `../lie-groups/lie_groups.ipynb` for a full treatment of $SO(3)$, $SE(3)$, adjoint representations, and the Baker–Campbell–Hausdorff formula.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SO(2) as a continuous group — exponential map and group structure
# ═══════════════════════════════════════════════════════════════════════════════

def R_SO2(theta):
    """SO(2) rotation matrix for angle theta (radians)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def so2_generator(omega):
    """Element of the Lie algebra so(2): omega * [[0,-1],[1,0]]."""
    return omega * np.array([[0, -1], [1, 0]])

def exp_so2(Omega):
    """Matrix exponential for so(2) element Omega (exact formula)."""
    omega = Omega[1, 0]   # extract scalar
    return R_SO2(omega)

def mat_exp_series(A, n_terms=20):
    """Matrix exponential via Taylor series: sum_k A^k / k!"""
    result = np.eye(A.shape[0])
    Ak     = np.eye(A.shape[0])
    fact   = 1.0
    for k in range(1, n_terms):
        Ak     = Ak @ A
        fact  *= k
        result += Ak / fact
    return result

# ── Verify exponential map ────────────────────────────────────────────────────
print("Exponential map verification for SO(2):")
for theta in [0, np.pi/6, np.pi/4, np.pi/2, np.pi, 2*np.pi/3]:
    Omega  = so2_generator(theta)
    R_exp  = mat_exp_series(Omega)
    R_exact= R_SO2(theta)
    ok     = np.allclose(R_exp, R_exact, atol=1e-10)
    print(f"  theta={np.degrees(theta):6.1f}°  exp(Omega) == R(theta): [{'PASS' if ok else 'FAIL'}]")

# ── Group homomorphism (R/2piZ, +) -> SO(2) ──────────────────────────────────
print("\nSO(2) group law — R(a) @ R(b) == R(a+b):")
test_pairs = [(np.pi/3, np.pi/6), (np.pi/2, np.pi/4), (1.2, 2.3)]
for a, b in test_pairs:
    lhs = R_SO2(a) @ R_SO2(b)
    rhs = R_SO2(a + b)
    ok  = np.allclose(lhs, rhs, atol=1e-12)
    print(f"  ({np.degrees(a):.1f}°, {np.degrees(b):.1f}°)  [{'PASS' if ok else 'FAIL'}]")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Visualization: SO(2) as unit circle + exponential map trajectories
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Left: SO(2) as the unit circle (manifold view) ────────────────────────────
ax = axes[0]
circle = plt.Circle((0, 0), 1, color=COLORS['blue'], fill=False, lw=2, label='$SO(2)$ manifold')
ax.add_patch(circle)

thetas_demo = np.linspace(0, 2*np.pi, 9, endpoint=False)
colors_demo = plt.cm.plasma(np.linspace(0.1, 0.9, len(thetas_demo)))

for i, theta in enumerate(thetas_demo):
    R     = R_SO2(theta)
    # The first column of R is where e_1 = [1,0] is mapped
    e1_img = R[:, 0]
    ax.annotate('', xy=e1_img, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=colors_demo[i], lw=2))
    ax.text(e1_img[0]*1.15, e1_img[1]*1.15,
            f'{np.degrees(theta):.0f}°', fontsize=7.5,
            ha='center', va='center', color=colors_demo[i])

ax.set_xlim(-1.55, 1.55)
ax.set_ylim(-1.55, 1.55)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$x$');  ax.set_ylabel('$y$')
ax.set_title('$SO(2)$ — Rotation group as the unit circle', fontsize=11, fontweight='bold')
ax.grid(alpha=0.2)

# ── Right: exp map — one-parameter subgroup from Lie algebra to Lie group ─────
ax2 = axes[1]
ts  = np.linspace(0, 2*np.pi, 300)
xs  = np.cos(ts)
ys  = np.sin(ts)
ax2.plot(xs, ys, color=COLORS['blue'], lw=2, label='$SO(2)$ group manifold')

# Draw exp(t*X) for a few values of t (one-parameter subgroup)
omega = 1.0
t_vals = np.linspace(0, 2*np.pi, 200)
exp_pts = np.array([mat_exp_series(so2_generator(omega * t)) @ np.array([1,0])
                    for t in t_vals])
ax2.plot(exp_pts[:,0], exp_pts[:,1], '--', color=COLORS['orange'],
         lw=2.5, label='exp$(t \\cdot X)$, one-parameter subgroup')

# Mark start
ax2.scatter([1], [0], color=COLORS['red'], s=100, zorder=5, label='Identity $I$')

# Lie algebra tangent at identity
ax2.annotate('', xy=(1 + 0, 0 + 0.45), xytext=(1, 0),
             arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2.5))
ax2.text(1.05, 0.25, '$X \\in \\mathfrak{so}(2)$\n(tangent at $e$)',
         fontsize=9, color=COLORS['green'])

ax2.set_xlim(-1.5, 1.7)
ax2.set_ylim(-1.4, 1.4)
ax2.set_aspect('equal')
ax2.axhline(0, color='gray', lw=0.5)
ax2.axvline(0, color='gray', lw=0.5)
ax2.set_xlabel('$x$');  ax2.set_ylabel('$y$')
ax2.set_title('Exponential map: $\\mathfrak{so}(2) \\to SO(2)$', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8, loc='lower left')
ax2.grid(alpha=0.2)

plt.suptitle('$SO(2)$ — From Lie Algebra to Lie Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Visualization: Symmetry group of an equilateral triangle  ≅  S_3 / D_3
# ═══════════════════════════════════════════════════════════════════════════════

def equilateral_triangle(center=(0,0), r=1.0, angle_offset=np.pi/2):
    """Return vertices of equilateral triangle centred at `center`."""
    angles = np.array([angle_offset + 2*np.pi*k/3 for k in range(3)])
    cx, cy = center
    return np.array([[cx + r*np.cos(a), cy + r*np.sin(a)] for a in angles])

# Symmetries of the triangle: 3 rotations + 3 reflections
sym_angles    = [0, 2*np.pi/3, 4*np.pi/3]  # rotations
sym_names_rot = ['$R_0$ (id)', '$R_{120}$', '$R_{240}$']
ref_axes      = [np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3]  # reflection axes
sym_names_ref = ['$\\sigma_1$', '$\\sigma_2$', '$\\sigma_3$']

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Symmetries of the Equilateral Triangle (Dihedral Group $D_3 \\cong S_3$)',
             fontsize=13, fontweight='bold')

V = equilateral_triangle(r=0.9)
vertex_labels = ['1', '2', '3']
vertex_colors_map = [COLORS['red'], COLORS['blue'], COLORS['green']]

def draw_triangle_sym(ax, V_orig, V_sym, title, perm, is_reflection=False, axis_angle=None):
    # Draw original (ghost)
    ghost = plt.Polygon(V_orig, fill=False, edgecolor='lightgray', lw=1.5, linestyle='--')
    ax.add_patch(ghost)
    # Draw transformed
    tri = plt.Polygon(V_sym, fill=True, facecolor='#E3F2FD', edgecolor=COLORS['blue'], lw=2)
    ax.add_patch(tri)
    # Vertex labels
    for i, (v, vc) in enumerate(zip(V_sym, vertex_colors_map)):
        ax.scatter(*v, color=vc, s=100, zorder=5)
        offset_sign = np.sign(v) if np.any(v != 0) else np.array([0.15, 0.15])
        ax.text(v[0]*1.18, v[1]*1.18, vertex_labels[perm[i]],
                fontsize=11, fontweight='bold', color=vc,
                ha='center', va='center')
    # Reflection axis
    if is_reflection and axis_angle is not None:
        L = 1.1
        ax.plot([-L*np.cos(axis_angle), L*np.cos(axis_angle)],
                [-L*np.sin(axis_angle), L*np.sin(axis_angle)],
                color=COLORS['orange'], lw=1.5, linestyle=':', label='reflection axis')
    ax.set_xlim(-1.4, 1.4);  ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=10, fontweight='bold')

# Row 0: rotations
for col, (angle, name) in enumerate(zip(sym_angles, sym_names_rot)):
    R  = R_SO2(angle)
    V2 = (R @ V.T).T
    # which vertex index of V2 is closest to each vertex of V? -> permutation
    perm_arr = [int(np.argmin(np.linalg.norm(V2 - v, axis=1))) for v in V]
    draw_triangle_sym(axes[0, col], V, V2, f'Rotation {name}', perm_arr)

# Row 1: reflections
def reflect_matrix(axis_angle):
    c, s = np.cos(2*axis_angle), np.sin(2*axis_angle)
    return np.array([[c, s], [s, -c]])

for col, (ax_ang, name) in enumerate(zip(ref_axes, sym_names_ref)):
    M  = reflect_matrix(ax_ang)
    V2 = (M @ V.T).T
    perm_arr = [int(np.argmin(np.linalg.norm(V2 - v, axis=1))) for v in V]
    draw_triangle_sym(axes[1, col], V, V2, f'Reflection {name}',
                     perm_arr, is_reflection=True, axis_angle=ax_ang)

plt.tight_layout()
plt.show()

print("D_3 ≅ S_3: The 6 symmetries of the equilateral triangle form a group isomorphic to S_3.")

## 10. Summary

### Key results covered

| Result | Statement |
|--------|----------|
| **Group axioms** | Closure, associativity, identity, inverse — the minimal structure for symmetry |
| **Latin square property** | Every element appears exactly once in each row/column of a Cayley table |
| **Lagrange's theorem** | $|H|$ divides $|G|$ for any $H \leq G$ (finite $G$) |
| **First isomorphism theorem** | $G/\ker\phi \cong \mathrm{im}\,\phi$ |
| **Cayley's theorem** | Every finite group embeds in some $S_n$ |
| **$A_n$ alternating group** | Even permutations form a normal subgroup of index 2 in $S_n$ |
| **Exponential map** | $\exp : \mathfrak{g} \to G$ lifts the Lie algebra to the Lie group |

### Group theory hierarchy

```
Semigroup  (closure + associativity)
  └─ Monoid  (+ identity)
       └─ Group  (+ inverses)
            ├─ Abelian group  (+ commutativity)  →  Z, Z_n, R
            └─ Non-abelian   →  S_n, GL(n), SO(3), SE(3)
                  └─ Lie group  (+ smooth manifold)  →  SO(n), SU(n), SE(n)
```

### Next steps

- **`../lie-groups/lie_groups.ipynb`** — $SO(3)$, $SE(3)$, adjoint, BCH formula
- **Ring and field theory** — add distributivity over two operations
- **Representation theory** — homomorphisms $G \to GL(V)$; characters, Schur's lemma
- **Galois theory** — subgroup lattices of $\mathrm{Gal}(E/F)$ classify field extensions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Summary table — groups studied in this notebook
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 75)
print(f"{'Group':<18} {'Order':>7} {'Abelian':>8} {'Cyclic':>8} {'# Subgroups':>12}")
print("=" * 75)

groups_summary = [
    (Z4,  True,  True),
    (V4,  True,  False),
    (Z5,  True,  True),
    (Z6,  True,  False),   # Z_6 is cyclic but has non-trivial subgroups
    (S3,  False, False),
    (S4,  False, False),
]

# Override cyclic flag for Z_6
cyclic_flags = [True, False, True, True, False, False]

for (G, _, __), cyc in zip(groups_summary, cyclic_flags):
    n_subs = len(G.subgroups()) if G.n <= 24 else 'N/A'
    ab_str = 'yes' if G.is_abelian() else 'no'
    cy_str = 'yes' if cyc else 'no'
    print(f"  {G.name:<16} {G.n:>7}  {ab_str:>8} {cy_str:>8} {str(n_subs):>12}")

print("=" * 75)
print("\nAll groups passed axiom verification  [PASS]")

## References

1. **Dummit, D. S. & Foote, R. M.** (2004). *Abstract Algebra* (3rd ed.). Wiley. — Chapters 1–4 (groups), Chapter 10 (modules), Chapter 14 (Galois theory).

2. **Artin, M.** (2011). *Algebra* (2nd ed.). Pearson. — Chapters 2–7 (groups through symmetry lens).

3. **Lang, S.** (2002). *Algebra* (3rd ed.). Springer GTM 211. — Rigorous graduate reference.

4. **Hall, B.** (2015). *Lie Groups, Lie Algebras, and Representations* (2nd ed.). Springer GTM 222. — Connects finite group theory to the Lie group setting; Chapter 1–2.

5. **Stillwell, J.** (2008). *Naive Lie Theory*. Springer. — Accessible introduction to the exponential map and $SO(n)$, $SU(n)$.

6. **Related notebooks in this collection:**
   - `../lie-groups/lie_groups.ipynb` — Lie groups and algebras in full generality
   - `../matrix-manifolds/matrix_manifolds.ipynb` — Riemannian geometry of matrix groups
   - `../../differential-geometry/differential_geometry.ipynb` — Smooth manifolds and tangent bundles